In [ ]:
# ============================================================
# =======================  CELL A  ============================
# ========== ROOT CELL — TTLs + QC + PORTABLE SPIKES =========
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

print("\n===== CELL A — ROOT PORTABLE EXTRACTION =====")

# ------------------------------------------------------------
# USER INPUTS  ###############################################
# ------------------------------------------------------------

SESSION_FOLDER = r"D:\Kevin\04032026"
NI_TTL_FOLDER = r"G:\Kevin\2026-03-04_08-31-33\Record Node 104\experiment1\recording1\events\NI-DAQmx-103.PXIe-6341\TTL"
PROBEA_PATH = r"G:\Kevin\2026-03-04_08-31-33\Record Node 101\experiment1\recording1\continuous\Neuropix-PXI-100.ProbeA\kilosort4"

CAMERA_TTL_PIN = 1
EXPECTED_RATE_HZ = 60
MISSING_TTL_THRESHOLD = 60
MIN_BLOCK_LENGTH = 1000

FILENAME_PATTERN = r"frame_(\d+)_cnt_(\d+)\.raw"

print("User inputs loaded successfully.")
print(f"SESSION_FOLDER: {SESSION_FOLDER}")
print(f"NI_TTL_FOLDER: {NI_TTL_FOLDER}")
print(f"PROBEA_PATH: {PROBEA_PATH}")
print(f"CAMERA_TTL_PIN: {CAMERA_TTL_PIN}")

# ------------------------------------------------------------
# UNIFIED ANALYSIS FOLDER (STATIC, SHARED ACROSS PIPELINES)
# ------------------------------------------------------------

def find_session_root(path):
    """
    Walk upward until we find the 'Record Node XXX' folder.
    """
    path = Path(path).resolve()
    while True:
        if "record node" in path.name.lower():
            return path
        parent = path.parent
        if parent == path:
            raise RuntimeError("Could not find 'Record Node' folder above given path")
        path = parent


def build_analysis_folder(session_root):
    """
    Build analysis folder using the known OpenEphys structure:
    session_date / Record Node XXX / experimentY / recordingZ / ...
    """

    # ---- Find experiment folder inside Record Node ----
    experiment_folders = [
        f for f in session_root.iterdir()
        if f.is_dir() and f.name.lower().startswith("experiment")
    ]
    if not experiment_folders:
        raise RuntimeError("No experiment folder found inside Record Node folder")
    experiment_folder = experiment_folders[0]

    # ---- Find recording folder inside experiment ----
    recording_folders = [
        f for f in experiment_folder.iterdir()
        if f.is_dir() and f.name.lower().startswith("recording")
    ]
    if not recording_folders:
        raise RuntimeError("No recording folder found inside experiment folder")
    recording_folder = recording_folders[0]

    # ---- Session date folder is parent of Record Node ----
    session_date_folder = session_root.parent

    session_date = session_date_folder.name
    experiment_name = experiment_folder.name
    recording_name = recording_folder.name

    # ---- Build analysis folder ----
    folder_name = f"{session_date}_{experiment_name}_{recording_name}_analysis"
    analysis_folder = session_root / folder_name
    analysis_folder.mkdir(exist_ok=True)

    # ---- Required subfolders ----
    subfolders = [
        "qc",
        "camera_alignment",
        "dlc_alignment",
        "behavior_ttls",
        "peak_detection",
        "metadata",
        "ttls",
        "spikes"
    ]
    for sf in subfolders:
        (analysis_folder / sf).mkdir(exist_ok=True)

    return analysis_folder


def get_analysis_folder(path_from_any_pipeline):
    session_root = find_session_root(path_from_any_pipeline)
    return build_analysis_folder(session_root)

# ------------------------------------------------------------
# AUTO-INCREMENTING FILE SAVE + LATEST-LOAD HELPERS
# ------------------------------------------------------------

def save_incrementing(folder, base_name, array):
    folder = Path(folder)
    folder.mkdir(exist_ok=True)

    # Try base_name.npy first
    path = folder / f"{base_name}.npy"
    if not path.exists():
        np.save(path, array)
        return path

    # Otherwise increment
    counter = 1
    while True:
        path = folder / f"{base_name}_{counter}.npy"
        if not path.exists():
            np.save(path, array)
            return path
        counter += 1

def load_latest(folder, base_name):
    folder = Path(folder)
    candidates = list(folder.glob(f"{base_name}*.npy"))
    if not candidates:
        raise FileNotFoundError(f"No files matching {base_name} in {folder}")
    latest = sorted(candidates, key=lambda p: p.stem)[-1]
    return np.load(latest, allow_pickle=True)

# ------------------------------------------------------------
# LOAD NI-DAQ TTL STREAM
# ------------------------------------------------------------

timestamps_ni = np.load(os.path.join(NI_TTL_FOLDER, "timestamps.npy"))
full_words_ni = np.load(os.path.join(NI_TTL_FOLDER, "full_words.npy"))

print(f"Loaded NI-DAQ data: {len(full_words_ni)} samples")

# ------------------------------------------------------------
# EXTRACT TTLs FOR ALL PINS (0–7)
# ------------------------------------------------------------

def extract_rising_edges(full_words, timestamps, pin):
    mask = 1 << pin
    bit_on = (full_words & mask) > 0
    bit_prev = np.roll(bit_on, 1)
    bit_prev[0] = False
    rising = (~bit_prev) & bit_on
    return timestamps[rising]

ttls_dir = analysis_folder / "ttls"

all_pin_ttls = {}
for pin in range(8):
    ttls = extract_rising_edges(full_words_ni, timestamps_ni, pin)
    all_pin_ttls[pin] = ttls
    save_incrementing(ttls_dir, f"TTL_Pin_{pin}", ttls)
    print(f"Saved {len(ttls)} TTLs for pin {pin}")

# Save NI timing metadata
ni_meta = {
    "pins": list(range(8)),
    "camera_ttl_pin_used_for_blocks": int(CAMERA_TTL_PIN),
    "ni_start_time": float(timestamps_ni[0]),
    "ni_end_time": float(timestamps_ni[-1]),
    "expected_rate_hz": EXPECTED_RATE_HZ,
    "missing_ttl_threshold": MISSING_TTL_THRESHOLD,
    "min_block_length": MIN_BLOCK_LENGTH,
}
with open(analysis_folder / "metadata" / "ttl_metadata.json", "w") as f:
    json.dump(ni_meta, f, indent=2)

print("Saved TTL metadata.")

# ------------------------------------------------------------
# TTL BLOCK DETECTION (CAMERA PIN)
# ------------------------------------------------------------

print("\n===== TTL BLOCK DETECTOR =====")

rising_times = all_pin_ttls[CAMERA_TTL_PIN]
print(f"Detected {len(rising_times)} rising edges on camera pin {CAMERA_TTL_PIN}")

ipi = np.diff(rising_times)
expected_ipi = 1.0 / EXPECTED_RATE_HZ
gap_threshold = expected_ipi * MISSING_TTL_THRESHOLD

gap_mask = ipi > gap_threshold
gap_indices = np.where(gap_mask)[0]

block_starts = np.insert(gap_indices + 1, 0, 0)
block_ends   = np.append(gap_indices, len(rising_times) - 1)

ttl_blocks = []
block_info = []

for b, (start, end) in enumerate(zip(block_starts, block_ends)):
    block_len = end - start + 1
    if block_len < MIN_BLOCK_LENGTH:
        print(f"Skipping block {b} (too short: {block_len})")
        continue

    block_times = rising_times[start:end+1]
    ttl_blocks.append(block_times)

    info = {
        "block_index": b,
        "ttl_start_index": int(start),
        "ttl_end_index": int(end),
        "n_ttls": int(block_len),
        "start_time_sec": float(block_times[0]),
        "end_time_sec": float(block_times[-1]),
        "duration_sec": float(block_times[-1] - block_times[0]),
        "gap_threshold_sec": float(gap_threshold),
        "camera_ttl_pin": int(CAMERA_TTL_PIN),
    }
    block_info.append(info)

    print(f"\nBlock {b}:")
    print(f"  TTLs: {block_len}")
    print(f"  Start: {block_times[0]:.3f}s")
    print(f"  End:   {block_times[-1]:.3f}s")

# Save block info
with open(analysis_folder / "metadata" / "block_info.json", "w") as f:
    json.dump(block_info, f, indent=2)

# Save TTL blocks
cam_align_dir = analysis_folder / "camera_alignment"
for b, block_times in enumerate(ttl_blocks):
    save_incrementing(cam_align_dir, f"ttl_times_block{b}", block_times)

# Plot full-session raster
plt.figure(figsize=(14, 3))
plt.scatter(rising_times, np.zeros_like(rising_times), s=2, c='k')
plt.title(f"Full-session TTL raster — camera pin {CAMERA_TTL_PIN}")
plt.xlabel("Time (s)")
plt.yticks([])
plt.show()

print("\n===== TTL BLOCK DETECTOR COMPLETE =====")

# ------------------------------------------------------------
# QC COMPUTATION (SPIKES + RATE MATRIX + NOISE UNITS)
# ------------------------------------------------------------

print("\n===== QC CHECK: Firing-rate matrix =====")

qc_dir = analysis_folder / "qc"
spikes_dir = analysis_folder / "spikes"

rate_path = qc_dir / "rate_matrix.npy"
good_units_path = qc_dir / "good_units.npy"
centers_path = qc_dir / "centers.npy"
duration_path = qc_dir / "recording_duration.json"
noise_units_path = qc_dir / "noise_units.npy"
noise_rate_path = qc_dir / "noise_rate_matrix.npy"

if rate_path.exists() and good_units_path.exists() and centers_path.exists() and duration_path.exists() and noise_units_path.exists() and noise_rate_path.exists():
    print("QC files already exist — skipping computation.")
else:
    print("QC files missing — computing...")

    # Load spike times
    spike_times = np.load(Path(PROBEA_PATH) / "spike_times.npy") / 30000.0
    spike_clusters = np.load(Path(PROBEA_PATH) / "spike_clusters.npy")

    # Load cluster info
    df_info = pd.read_csv(Path(PROBEA_PATH) / "cluster_info.tsv", sep="\t")

    cluster_id_col = "id"
    group_col = "group"

    # ---------------- GOOD UNITS ----------------
    good_units = df_info[df_info[group_col] == "good"][cluster_id_col].values
    np.save(good_units_path, good_units)
    print(f"Loaded {len(good_units)} good units.")

    # Save portable spike times for good units
    for unit in good_units:
        unit_spikes = spike_times[spike_clusters == unit]
        np.save(spikes_dir / f"unit_{unit}.npy", unit_spikes)

    # ---------------- NOISE UNITS ----------------
    noise_units = df_info[df_info[group_col] == "noise"][cluster_id_col].values
    print(f"Found {len(noise_units)} noise units total.")

    np.random.seed(0)
    if len(noise_units) > 20:
        sampled_noise_units = np.random.choice(noise_units, size=20, replace=False)
    else:
        sampled_noise_units = noise_units

    sampled_noise_units = np.array(sampled_noise_units)
    np.save(noise_units_path, sampled_noise_units)
    print(f"Using {len(sampled_noise_units)} sampled noise units.")

    # Save portable spike times for noise units
    for unit in sampled_noise_units:
        unit_spikes = spike_times[spike_clusters == unit]
        np.save(spikes_dir / f"noise_unit_{unit}.npy", unit_spikes)

    # ---------------- TIME BINS ----------------
    recording_duration = float(spike_times.max())
    bin_size = 0.02
    bins = np.arange(0, recording_duration + bin_size, bin_size)
    centers = bins[:-1] + bin_size/2

    np.save(centers_path, centers)
    with open(duration_path, "w") as f:
        json.dump({"recording_duration": recording_duration}, f)

    # ---------------- RATE MATRIX (GOOD UNITS) ----------------
    rate_matrix = np.zeros((len(good_units), len(centers)), float)

    for i, unit in enumerate(good_units):
        unit_spikes = spike_times[spike_clusters == unit]
        hist, _ = np.histogram(unit_spikes, bins=bins)
        rate_matrix[i, :] = gaussian_filter1d(hist.astype(float), sigma=2)

    np.save(rate_path, rate_matrix)
    print(f"Saved rate_matrix → {rate_path}")

    # ---------------- RATE MATRIX (NOISE UNITS) ----------------
    noise_rate_matrix = np.zeros((len(sampled_noise_units), len(centers)), float)

    for i, unit in enumerate(sampled_noise_units):
        unit_spikes = spike_times[spike_clusters == unit]
        hist, _ = np.histogram(unit_spikes, bins=bins)
        noise_rate_matrix[i, :] = gaussian_filter1d(hist.astype(float), sigma=2)

    np.save(noise_rate_path, noise_rate_matrix)
    print(f"Saved noise_rate_matrix → {noise_rate_path}")

print("\n===== QC COMPLETE =====\n")